In [1]:
import pandas as pd

strongs = pd.read_pickle("pickles/strongs.pickle")
sept = pd.read_pickle("pickles/sept.pickle")
tisch = pd.read_pickle("pickles/tisch.pickle")
bible = pd.concat([sept, tisch], axis=0)
bible

,text,str,verse,chapter,book,rmac
0,εν,1722,1,1,1,prep
1,αρχη,746,1,1,1,n-dsf
2,εποιησεν,4160,1,1,1,v-aai-3s
3,ο,3588,1,1,1,t-nsm
4,θεος,2316,1,1,1,n-nsm
...,...,...,...,...,...,...
137520,τοῦ,3588,21,22,66,t-gsm
137521,κυρίου,2962,21,22,66,n-gsm
137522,ἰησοῦ,2424,21,22,66,n-gsm
137523,μετὰ,3326,21,22,66,prep


In [2]:
# nouns, verbs, and adjectives
interesting_words = bible[
    bible["rmac"].str.startswith("n-")
    | bible["rmac"].str.startswith("v-")
    | bible["rmac"].str.startswith("a-")
    | (bible["rmac"] == "adv")
]

verses = [
    verse["str"]
    for id_, verse in interesting_words.groupby(["book", "chapter", "verse"])
]

In [3]:
strongs.loc[verses[0][~verses[0].str.contains("C")], "def"]

str
746     beginning, corner, (at the, the) first (estate...
4160    abide, + agree, appoint, X avenge, + band toge...
2316                    X exceeding, God, god(-ly, -ward)
3772                                air, heaven(-ly), sky
1065                and besides, doubtless, at least, yet
Name: def, dtype: object

Following https://radimrehurek.com/gensim/auto_examples/tutorials/run_lda.html

In [4]:
from gensim.models import Phrases

In [5]:
bigram = Phrases(verses, min_count=10)
verses = bigram[verses]

In [6]:
from gensim.corpora import Dictionary

dictionary = Dictionary(verses)
# filter lemmas that occur in less than 5 verses, or in more than half the books
dictionary.filter_extremes(no_below=10, no_above=0.5)

From 

In [7]:
from gensim.models.ldamodel import LdaModel
from gensim.models.coherencemodel import CoherenceModel

In [8]:
corpus = [dictionary.doc2bow(verse) for verse in verses]

In [10]:
new = pd.DataFrame(columns=["ntopics", "alpha", "eta", "coherence"])
for num_topics in [10, 15, 30, 50, 80, 100]:
    for alpha in [1 / num_topics, 0.1, 1, "auto"]:
        for eta in [0.001, 0.01, 0.1, "auto"]:
            model = LdaModel(
                corpus=corpus,
                id2word=dictionary,
                alpha=alpha,
                eta=eta,
                num_topics=num_topics,
                passes=5,
                iterations=100,
            )
            cm = bdsCoherenceModel(
                model=model, texts=verses, dictionary=dictionary, coherence="c_v"
            )
            new.loc[len(new)] = [num_topics, alpha, eta, cm.get_coherence()]

In [ ]:
new.sort_values("coherence", ascending=False)

In [13]:
df = pd.read_csv("params.txt", delimiter=" ")

In [ ]:
df.sort_values("coherence", ascending=False)

In [20]:
for highv in [51, 53, 52]:
    row = df.iloc[highv]
    model = LdaModel(
        corpus=corpus,
        id2word=dictionary,
        alpha="auto" if row["alpha"] == "auto" else float(row["alpha"]),
        eta="auto" if row["eta"] == "auto" else float(row["eta"]),
        num_topics=row["ntopics"],
        passes=20,
        iterations=300,
    )
    cm = CoherenceModel(
        model=model, texts=verses, dictionary=dictionary, coherence="c_v"
    )
    print(row, cm.get_coherence())

ntopics           80
alpha            0.1
eta             0.01
coherence    0.59293
Name: 51, dtype: object 0.6071294262076086
ntopics            80
alpha             0.1
eta              auto
coherence    0.580056
Name: 53, dtype: object 0.6040165988349887
ntopics            80
alpha             0.1
eta               0.1
coherence    0.570591
Name: 52, dtype: object 0.5616416088417482


In [21]:
model = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    alpha=0.1,
    eta="auto",
    num_topics=80,
    passes=30,
    iterations=3000,
)